In [ ]:
VERSION = "1.0.1_sextractor"

# lib

In [ ]:
from pathlib import Path

from jax import numpy
from polars import DataFrame, read_csv
from tqdm import tqdm
from utils import load_and_slice_csv, ratio_angle_to_e1e2
from utils_plot import (
    check_psf,
    final_results,
    hist_gal,
    histo_err,
    plot_sersic,
    plot_vignettes,
    print_fits,
)

ALL_DATASETS_DIR = Path("/net/GECO/pisco/PIX2SHEAR/DATASETS/pisco_euclid")
RESULTS_SEXTRACTOR_DIR = Path("/net/GECO/pisco/PIX2SHEAR/RESULTS_SEXTRACTOR")
DATASET_DIR = ALL_DATASETS_DIR / VERSION
IMAGES_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "IMAGES"
VIGNET_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "VIGNET"
RESULTS_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "RESULTS"
PLOTS_DIR = RESULTS_SEXTRACTOR_DIR / VERSION / "PLOTS"
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

# sersic

In [ ]:
X_min, X_max = -1.8, 1.5
plot_sersic(X_min, X_max, PLOTS_DIR)
del X_min, X_max

# psf

In [ ]:
check_psf(ALL_DATASETS_DIR / "PSF" / "airy.psf.fits", PLOTS_DIR)

# images

In [ ]:
CHOICE = "simu_00002"


print_fits(
    [
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        DATASET_DIR / CHOICE,
        IMAGES_DIR / f"fitting_{CHOICE}_1",
        IMAGES_DIR / f"residual_{CHOICE}_1",
        IMAGES_DIR / f"segmentation_{CHOICE}",
        IMAGES_DIR / f"background_{CHOICE}",
    ],
    ["image", "fitting", "image", "fitting", "residual", "segmentation", "background"],
    PLOTS_DIR,
)

# vignettes

In [ ]:
CHOICE = "simu_00001"

X = read_csv(f"{RESULTS_DIR / CHOICE}.csv")["x"]
Y = read_csv(f"{RESULTS_DIR / CHOICE}.csv")["y"]
VIGNETTES = sorted(list((VIGNET_DIR / CHOICE).glob("*.fits")))

plot_vignettes(VIGNETTES, X, Y, PLOTS_DIR)
del VIGNETTES, X, Y

# results

In [ ]:
file_path = RESULTS_DIR / "pred_avg_ellipticies.csv"


if file_path.exists():
    NB_TOTAL = len(sorted(list((RESULTS_DIR).glob("simu_*.csv"))))

else:
    RESULTS_FILES = sorted(list((RESULTS_DIR).glob("simu_*.csv")))
    NB_TOTAL = len(RESULTS_FILES)
    NB_FAKE_SEG = 0
    e1, e2, avg_e1, avg_e2, ratio, angle, gal_detect = [], [], [], [], [], [], []
    for sample in tqdm(RESULTS_FILES, total=NB_TOTAL):
        df = read_csv(sample)
        avg_e1.append(df["e1"].mean())
        avg_e2.append(df["e2"].mean())
        e1.append(df["e1"].to_numpy())
        e2.append(df["e2"].to_numpy())
        ratio.append(df["ratio"].to_numpy())
        angle.append(df["angle"].to_numpy())
        gal_detect.append(df.shape[0])
        if df.shape[0] != 10:
            NB_FAKE_SEG += 1

    results = DataFrame({"avg_e1": avg_e1, "avg_e2": avg_e2}, schema=["avg_e1", "avg_e2"])
    results.write_csv(file_path)
    pred_avg_e1, pred_avg_e2 = results

    # check
    e1 = numpy.concatenate(e1)
    e2 = numpy.concatenate(e2)
    ratio = numpy.concatenate(ratio)
    angle = numpy.concatenate(angle)
    recons_e1, recons_e2 = ratio_angle_to_e1e2(ratio, angle)

    histo_err(e1 - recons_e1, e2 - recons_e2, PLOTS_DIR, bins=10, temp=0.0001)
    hist_gal(gal_detect, NB_FAKE_SEG, PLOTS_DIR)

    del RESULTS_FILES, results, df, avg_e1, avg_e2, e1, e2, ratio, angle, recons_e1, recons_e2

# postprocess

In [ ]:
pred_avg_e = load_and_slice_csv(RESULTS_DIR / "pred_avg_ellipticies.csv", NB_TOTAL)
true_avg_e = load_and_slice_csv(DATASET_DIR / "true_avg_ellipticies.csv", NB_TOTAL)
true_shears = load_and_slice_csv(DATASET_DIR / "true_avg_shears.csv", NB_TOTAL)
print(pred_avg_e.shape, true_avg_e.shape, true_shears.shape)

for i in range(2):
    pred_avg_e = pred_avg_e[numpy.abs(true_shears[:, i]) <= 0.05]
    true_avg_e = true_avg_e[numpy.abs(true_shears[:, i]) <= 0.05]
    true_shears = true_shears[numpy.abs(true_shears[:, i]) <= 0.05]
print(pred_avg_e.shape, true_avg_e.shape, true_shears.shape)

In [ ]:
final_results(true_avg_e, pred_avg_e, true_shears, PLOTS_DIR)